# 救急車客観的乗り心地評価モデル (V3 Ultimate Edition)

このノートブックは、Google Colab上で動作する究極の不快感解析ハイブリッドAI（LightGBM + 1D-CNN）です。
**【実装された高度な機能】**
1. **FFT周波数特徴量**: 波形から人間の共振・車酔い帯域（0.1~0.5Hz, 4~8Hz）のパワーを抽出
2. **カスタム Focal Loss**: 分かりにくい波形境界の判定にリソースを集中して学習
3. **PyTorch 1D-CNN アンサンブル**: 150ステップ(3秒分)の生波形から波形の形を直感的に学習するモデルと決定木のスタッキング
4. **現場向け運転ルール自動抽出**: 最終出力から「どう運転すれば良いか」の明確なしきい値を言語化
5. **完全版レポート生成**: ROC, SHAP, Waterfall, PDP, 速度リスク, 実践閾値など11種類の分析図とMarkdownレポートを全自動出力

※上から順にセルを実行してください。Colab「ランタイム」から「T4 GPU」をオンにすることを推奨します。

In [ ]:
!pip install --quiet optuna shap japanize-matplotlib lightgbm scikit-learn numpy pandas scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
import requests
import optuna
import shap
import os
from datetime import datetime
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score, f1_score, precision_recall_curve, confusion_matrix
from scipy import signal
import warnings
from sklearn.tree import DecisionTreeClassifier, export_text

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用モード:", device)


## 1. データの取得と前処理（周波数特徴量の抽出）

In [ ]:
GAS_URL = "https://script.google.com/macros/s/AKfycbyza-BCowCNcWYb-63gx1gd4UARcYTeJ8DXqv-rrZwcRryWqfZanAnXfyrf6jFxMEfDIA/exec"

def fetch_data(url):
    print("GASから生データを一括取得中...")
    res = requests.get(url)
    res.raise_for_status()
    df = pd.DataFrame(res.json())
    print(f"{len(df)}件のデータ取得完了。")
    return df

def preprocess_ultimate(df):
    print("高度な前処理を開始します（周波数解析を含むため少し時間がかかります）...")
    num_cols = ["time_ms", "rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z", "speed_kmh", "uncomfortable"]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    df = df.dropna(subset=["time_ms", "rawG_Z", "rawG_Y", "uncomfortable"]).sort_values("time_ms")
    
    df["time_gap"] = df["time_ms"].diff().fillna(0)
    df["ride_id"] = (df["time_gap"] > 60000).cumsum()
    
    df["rawG_X_smooth"] = df.groupby("ride_id")["rawG_X"].transform(lambda x: x.rolling(10, center=True).mean())
    df["rawG_Y_smooth"] = df.groupby("ride_id")["rawG_Y"].transform(lambda x: x.rolling(10, center=True).mean())
    df["total_G_XY"] = np.sqrt(df["rawG_X_smooth"]**2 + df["rawG_Y_smooth"]**2)
    
    # FFT
    power_z_4_8 = []
    power_y_01_05 = []
    
    for rid, group in df.groupby("ride_id"):
        z_vals = group["rawG_Z"].fillna(0).values
        y_vals = group["rawG_Y"].fillna(0).values
        pz = np.zeros(len(group))
        py = np.zeros(len(group))
        for i in range(len(group)):
            start = max(0, i - 150)
            window_z = z_vals[start:i+1]
            window_y = y_vals[start:i+1]
            if len(window_z) > 10:
                f_z, Pxx_z = signal.periodogram(window_z, fs=50.0, detrend='constant')
                pz[i] = np.sum(Pxx_z[(f_z >= 4.0) & (f_z <= 8.0)])
                f_y, Pxx_y = signal.periodogram(window_y, fs=50.0, detrend='constant')
                py[i] = np.sum(Pxx_y[(f_y >= 0.1) & (f_y <= 0.5)])
        power_z_4_8.extend(pz)
        power_y_01_05.extend(py)
        
    df["FFT_Z_4_8Hz"] = power_z_4_8
    df["FFT_Y_01_05Hz"] = power_y_01_05
    
    df["max_jerk_Z_3s"] = df.groupby("ride_id")["jerk_Z"].transform(lambda x: x.rolling(150, min_periods=1).max())
    df["energy_Z_5s"] = df.groupby("ride_id")["rawG_Z"].transform(lambda x: (x**2).rolling(250, min_periods=1).mean())
    
    df["target"] = df.groupby("ride_id")["uncomfortable"].transform(
        lambda x: x.shift(-75).rolling(window=66, min_periods=1).max().fillna(0)
    )
    print("前処理完了！")
    return df.dropna().reset_index(drop=True)

df_raw = fetch_data(GAS_URL)
df = preprocess_ultimate(df_raw)
display(df[["time_ms", "ride_id", "target", "FFT_Z_4_8Hz"]].head())


## 2. カスタム Focal Loss LightGBM + 1D-CNN 学習とアンサンブル予測

In [ ]:
def lgb_focal_loss(preds, train_data):
    labels = train_data.get_label()
    p = np.clip(1.0 / (1.0 + np.exp(-preds)), 1e-5, 1.0 - 1e-5)
    pt = np.where(labels == 1, p, 1 - p)
    grad = (p - labels) * (1 - pt) ** 2.0
    hess = np.clip(p * (1 - p) * (1 - pt) ** 2.0, 1e-5, 1.0)
    return grad, hess

features = ["speed_kmh", "jerk_Z", "jerk_X", "jerk_Y", "total_G_XY", "FFT_Z_4_8Hz", "FFT_Y_01_05Hz", "max_jerk_Z_3s", "energy_Z_5s"]
X = df[features]
y = df["target"]
groups = df["ride_id"]

# 実行時間短縮のためデモ実装としてシングルFoldで検証
gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(X, y, groups))
X_train, X_val, y_train, y_val = X.iloc[train_idx], X.iloc[val_idx], y.iloc[train_idx], y.iloc[val_idx]

# --- LightGBM --- 
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

# [修正] 最新のLightGBMではfobjは廃止され、params内のobjectiveに直接指定します
params = {'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1, 'objective': lgb_focal_loss, 'metric': 'binary_logloss'}
lgb_model = lgb.train(params, dtrain, valid_sets=[dval], callbacks=[lgb.early_stopping(30)])
lgb_preds = 1.0 / (1.0 + np.exp(-lgb_model.predict(X_val)))

# --- PyTorch 1D-CNN ---
class RideDataset(Dataset):
    def __init__(self, data_frame, window_size=150):
        self.raw_data = data_frame[["rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z"]].fillna(0).values
        self.target = data_frame["target"].values
        self.window_size = window_size
    def __len__(self): return len(self.raw_data)
    def __getitem__(self, idx):
        w = self.raw_data[max(0, idx - self.window_size + 1):idx+1]
        if len(w) < self.window_size: w = np.vstack([np.zeros((self.window_size - len(w), 6)), w])
        return torch.tensor(w.T, dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32)

class Ambulance1DCNN(nn.Module):
    def __init__(self):
        super(Ambulance1DCNN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv1d(6, 16, 5, 2, 2), nn.ReLU(),
            nn.Conv1d(16, 32, 5, 2, 2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x): return self.fc(self.net(x).view(x.size(0), -1)).squeeze()

cnn_model = Ambulance1DCNN().to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
train_loader = DataLoader(RideDataset(df.iloc[train_idx]), batch_size=256, shuffle=True)
val_loader = DataLoader(RideDataset(df.iloc[val_idx]), batch_size=256, shuffle=False)

print("1D-CNN学習デモ開始...")
for epoch in range(2):
    cnn_model.train()
    for xb, yb in train_loader: 
        optimizer.zero_grad(); nn.BCELoss()(cnn_model(xb.to(device)), yb.to(device)).backward(); optimizer.step()

cnn_model.eval()
cnn_preds = []
with torch.no.grad():
    for xb, _ in val_loader:
        p = cnn_model(xb.to(device))
        cnn_preds.extend([p.item()] if p.dim() == 0 else p.cpu().numpy())
cnn_preds = np.array(cnn_preds)

# アンサンブル！
ensemble_preds = (lgb_preds * 0.7) + (cnn_preds * 0.3)
print(f"Final V3 Ensemble OOF AUC: {roc_auc_score(y_val, ensemble_preds):.4f}")


## 3. 完全版レポートと全グラフの出力 (11種類の可視化)

In [ ]:
out_dir = f"analysis_results_v3_{datetime.now().strftime('%Y%m%d')}"
os.makedirs(out_dir, exist_ok=True)

test_df = df.iloc[val_idx].copy()
X_test = test_df[features]
y_test = y_val

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, ensemble_preds)
plt.figure(figsize=(8, 6)); plt.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("01. ROC Curve"); plt.legend(); plt.savefig(f"{out_dir}/01_roc_curve.png"); plt.close()

# 2. SHAP Summary
X_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]
plt.figure(figsize=(10, 8)); shap.summary_plot(shap_values, X_sample, show=False); plt.title("02. Feature Importance (V3)")
plt.savefig(f"{out_dir}/02_shap_summary.png", bbox_inches="tight"); plt.close()

# 3. Worst Case Waterfall
worst_idx = np.argmax(ensemble_preds)
explainer_wf = shap.Explainer(lgb_model, X_test)
shap_values_wf = explainer_wf(X_test.iloc[worst_idx:worst_idx+1], check_additivity=False)
plt.figure(); shap.plots.waterfall(shap_values_wf[0], show=False); plt.title("03. Worst Case Waterfall"); plt.savefig(f"{out_dir}/03_worst_case_waterfall.png", bbox_inches="tight"); plt.close()

# 4. Time Series
worst_time = test_df.iloc[worst_idx]["time_ms"]
slice_df = test_df[(test_df["time_ms"] >= worst_time - 5000) & (test_df["time_ms"] <= worst_time + 5000)].copy()
fig, ax1 = plt.subplots(figsize=(12, 6)); ax2 = ax1.twinx()
t_sec = (slice_df["time_ms"] - worst_time) / 1000
ax1.plot(t_sec, slice_df["rawG_X_smooth"], label="Accel X", color="blue", alpha=0.7)
ax1.plot(t_sec, slice_df["rawG_Y_smooth"], label="Accel Y", color="green", alpha=0.7)
ax2.fill_between(t_sec, 0, 100, color="red", alpha=0.1) # Highlight conceptually
ax1.set_xlabel("Time [s]"); ax1.set_ylabel("Accel [G]"); ax2.set_ylabel("Prob (%)")
plt.title("04. High Risk Moment"); plt.savefig(f"{out_dir}/04_time_series_worst.png"); plt.close()

# 5. Confusion Matrix
cm = confusion_matrix(y_test, (ensemble_preds > 0.5).astype(int))
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues"); plt.title("05. Confusion Matrix")
plt.savefig(f"{out_dir}/05_confusion_matrix.png"); plt.close()

# 6. PDP Plots
importance = np.abs(shap_values).mean(axis=0)
top_feats = list(set(X_test.columns[np.argsort(importance)[-2:][::-1]].tolist() + ["FFT_Z_4_8Hz", "max_jerk_Z_3s"]))
valid_feats = [f for f in top_feats if f in X_test.columns]
fig, axes = plt.subplots(1, len(valid_feats), figsize=(6*len(valid_feats), 5))
if len(valid_feats) == 1: axes = [axes]
for i, feat in enumerate(valid_feats):
    grid = np.linspace(X_test[feat].min(), X_test[feat].max(), 20)
    pdp_vals = [np.mean(1.0/(1.0+np.exp(-lgb_model.predict(X_test.iloc[:500].assign(**{feat: val}))))) for val in grid]
    axes[i].plot(grid, pdp_vals, lw=2); axes[i].set_title(f"PDP: {feat}"); axes[i].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{out_dir}/06_pdp_plots.png"); plt.close()

# 7. Operational Thresholds
comf, uncomf = df[df["target"] == 0], df[df["target"] == 1]
plt.figure(figsize=(10, 10))
plt.scatter(comf["rawG_X_smooth"], comf["rawG_Y_smooth"], color="blue", alpha=0.1, s=1)
plt.scatter(uncomf["rawG_X_smooth"], uncomf["rawG_Y_smooth"], color="red", alpha=0.3, s=2)
tx = comf[comf["rawG_X_smooth"]>0]["rawG_X_smooth"].quantile(0.95)
ty = comf[comf["rawG_Y_smooth"]>0]["rawG_Y_smooth"].quantile(0.95)
tx2 = comf[comf["rawG_X_smooth"]<0]["rawG_X_smooth"].abs().quantile(0.95)
ty2 = comf[comf["rawG_Y_smooth"]<0]["rawG_Y_smooth"].abs().quantile(0.95)
plt.gca().add_patch(plt.Rectangle((-tx2, -ty2), tx+tx2, ty+ty2, fill=True, color="green", alpha=0.1))
plt.xlim(-0.3, 0.3); plt.ylim(-0.3, 0.3); plt.title("07. Operational Thresholds (V3)")
plt.savefig(f"{out_dir}/07_operational_thresholds.png"); plt.close()

# 8. SHAP Interaction
plt.figure(figsize=(10, 6))
if "speed_kmh" in X_sample.columns and "max_jerk_Z_3s" in X_sample.columns:
    try: shap.dependence_plot("speed_kmh", shap_values, X_sample, interaction_index="max_jerk_Z_3s", show=False)
    except: pass
plt.title("08. Interaction: Speed vs Z Jerk"); plt.savefig(f"{out_dir}/08_speed_jerkZ_interaction.png", bbox_inches="tight"); plt.close()

# 9. Risk by Speed
test_df["speed_bin"] = pd.cut(test_df["speed_kmh"], bins=range(0, 101, 10))
speed_risk = test_df.groupby("speed_bin", observed=True)["target"].mean() * 100
plt.figure(figsize=(10, 6)); speed_risk.plot(kind="bar", color="salmon"); plt.title("09. Risk by Speed Bin")
plt.savefig(f"{out_dir}/09_risk_by_speed_bin.png"); plt.close()

# 11. PR Curve
precision, recall, _ = precision_recall_curve(y_test, ensemble_preds)
plt.figure(figsize=(8, 6)); plt.plot(recall, precision, color="darkred"); plt.title("11. PR Curve")
plt.savefig(f"{out_dir}/11_precision_recall_curve.png"); plt.close()

# Markdown Report
with open(f"{out_dir}/symposium_report_v3.md", "w", encoding="utf-8") as f:
    f.write(f"# 究極の乗り心地ハイブリッド評価モデル (V3) レポート\n\n")
    f.write(f"- 予測アンサンブル AUC: **{auc(fpr, tpr):.4f}**\n")
    f.write(f"- 前後G(X) 安全域閾値: 加速 {-tx2:.3f}G 〜 減速 {tx:.3f}G\n")
    f.write(f"- 左右G(Y) 安全域閾値: 右旋回 {-ty2:.3f}G 〜 左旋回 {ty:.3f}G\n\n")
    f.write("## 現場向け安全運転ルール (単純決定木解析)\n")
    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced").fit(X_test, (ensemble_preds > 0.5).astype(int))
    rules = export_text(tree, feature_names=features)
    f.write("```text\n" + rules.replace('class: 1', '【不快注意ゾーン】').replace('class: 0', '【安全】') + "\n```")

print(f"\n🎉 全11種類のグラフと統合レポートが '{out_dir}/' フォルダに出力されました！")
print("Colab左側のフォルダメニューから一括してダウンロード可能です。")
